# EDA explicado — Federico
Cada sección sigue pregunta → cálculo → interpretación → decisión. Las asociaciones no demuestran causalidad. Este EDA usa el archivo completo para descripción; no debe utilizarse para ajustar modelos mirando el test. La procedencia y generación de las variables añadidas siguen pendientes.

## Procedencia y alcance del análisis
Según el README del equipo (revisión d2f501b), los 10.000 registros y 35 variables fueron generados sintéticamente desde cero en Python; IBM Telco Customer Churn fue solo una referencia conceptual y sus registros no se incorporaron. El equipo declara semilla 42, reglas aleatorias y una función logística para Churn.

**Cómo interpretar todas las conclusiones siguientes:** describen este conjunto simulado, no clientes reales. Las asociaciones pueden reflejar las reglas del generador y no demuestran causalidad ni eficacia de retención. Los cálculos existentes no cambian por esta aclaración.

Falta obtener y verificar el script original, sus fórmulas, autoría y licencia. La semilla sola no permite reproducir el CSV. Ver [dataset-card.md](../dataset-card.md).


## 1. Comprender la unidad de análisis
Comprobamos tamaño y ejemplos. Una fila corresponde a un identificador de cliente, pero no conocemos la fecha ni la ventana de observación.

In [1]:
from telecom_churn.data import load_data
import pandas as pd
import plotly.express as px
# No modificamos el CSV: la etiqueta auxiliar permite calcular tasas mediante la media.
df = load_data()
df['abandono'] = df.Churn.eq('Yes').astype(int)
print('Dimensiones originales:', df.drop(columns='abandono').shape)
display(df.head())

Dimensiones originales: (10000, 35)


,CustomerID,Age,SeniorCitizen,Gender,Partner,Dependents,TenureMonths,TenureCohort,PhoneService,MultipleLines,...,Complaints,CustomerContacts,ServiceOutages,NPS,CSAT,EstimatedCLV,AvgTicketCost,Churn,ChurnReason,abandono
0,TELCO-00001,46,0,M,No,Yes,72,61-72 months,Yes,Yes,...,0,4,0,8.0,5.0,8126.75,0.00,No,No churn,0
1,TELCO-00002,37,0,M,No,Yes,72,61-72 months,Yes,Yes,...,0,4,0,8.0,3.0,6655.91,0.93,No,No churn,0
2,TELCO-00003,48,0,F,Yes,No,35,25-36 months,Yes,No,...,0,1,1,5.0,3.0,3417.06,0.00,Yes,Poor customer service,1
3,TELCO-00004,60,1,F,No,No,53,49-60 months,Yes,No,...,0,4,0,10.0,4.0,5704.34,1.95,No,No churn,0
4,TELCO-00005,36,0,F,No,No,35,25-36 months,Yes,Yes,...,1,3,0,7.0,3.0,3916.51,0.83,Yes,Poor customer service,1


**Conclusión:** tenemos 10.000 registros y 35 columnas originales. La columna auxiliar `abandono` no es una característica nueva para entrenar. Sin fechas no podemos afirmar que predecimos la baja a 30 o 90 días.

## 2. Calidad: faltantes y duplicados
Cuantificamos qué falta antes de decidir cómo tratarlo. Un valor ausente no equivale a cero.

In [2]:
# Reportamos tanto faltantes por columna como clientes afectados; no se suman como personas distintas.
missing = df.isna().sum().to_frame('faltantes')
missing['porcentaje'] = missing.faltantes / len(df) * 100
display(missing.query('faltantes > 0'))
print('Filas afectadas:', df.isna().any(axis=1).sum())
print('IDs duplicados:', df.CustomerID.duplicated().sum())
print('Duplicados sin ID:', df.drop(columns='CustomerID').duplicated().sum())

,faltantes,porcentaje
PaymentMethod,150,1.5
MonthlyCharges,150,1.5
NPS,150,1.5
CSAT,150,1.5


Filas afectadas: 588
IDs duplicados: 0
Duplicados sin ID: 0


**Conclusión:** cuatro columnas tienen 150 faltantes cada una (1,5%): PaymentMethod, MonthlyCharges, NPS y CSAT. Afectan a 588 clientes (5,88%); algunas ausencias coinciden en la misma fila. No hay duplicados. **Decisión:** conservar filas e imputar dentro del pipeline, ajustando mediana/moda solo con entrenamiento.

## 3. Distribución del abandono
Medimos el desequilibrio de clases para entender por qué la exactitud por sí sola no basta.

In [3]:
counts = df.Churn.value_counts().rename_axis('Churn').reset_index(name='clientes')
display(counts)
px.bar(counts, x='Churn', y='clientes', title='Clientes que permanecen y abandonan').show()

,Churn,clientes
0,No,5831
1,Yes,4169


**Conclusión:** 4.169 abandonos (41,69%) frente a 5.831 permanencias (58,31%). Predecir siempre permanencia ya acertaría 58,31% del archivo, pero no detectaría ningún abandono. **Decisión:** comparar con baseline y evaluar recall, precisión, F1 y ROC-AUC.

## Abandono según Contract
**Pregunta:** ¿la proporción de abandono cambia entre los grupos? Calculamos tamaño y tasa; no confundimos número de abandonos con riesgo relativo del grupo.

In [4]:
# dropna=False conserva a quienes no tienen información en esta variable.
summary = df.groupby('Contract', dropna=False).abandono.agg(['size', 'mean']).reset_index()
summary = summary.rename(columns={'size':'clientes', 'mean':'tasa'})
display(summary)
px.bar(summary, x='Contract', y='tasa', hover_data=['clientes'], title='Abandono según Contract', labels={'tasa':'Proporción de abandono'}).show()

,Contract,clientes,tasa
0,Month-to-month,5526,0.501267
1,One year,2453,0.325316
2,Two year,2021,0.297378


**Conclusión:** Entre grupos de al menos 100 clientes, Month-to-month presenta 50.13% de abandono (5526 clientes) y Two year presenta 29.74% (2021 clientes). Los grupos pequeños requieren cautela y las diferencias no prueban causalidad. **Decisión:** mantener esta variable como candidata y comprobar su aporte en validación, considerando posibles relaciones con otras variables.

## Abandono según InternetService
**Pregunta:** ¿la proporción de abandono cambia entre los grupos? Calculamos tamaño y tasa; no confundimos número de abandonos con riesgo relativo del grupo.

In [5]:
# dropna=False conserva a quienes no tienen información en esta variable.
summary = df.groupby('InternetService', dropna=False).abandono.agg(['size', 'mean']).reset_index()
summary = summary.rename(columns={'size':'clientes', 'mean':'tasa'})
display(summary)
px.bar(summary, x='InternetService', y='tasa', hover_data=['clientes'], title='Abandono según InternetService', labels={'tasa':'Proporción de abandono'}).show()

,InternetService,clientes,tasa
0,DSL,2709,0.367294
1,Fiber,4239,0.510262
2,Mobile,2092,0.357553
3,No Internet,960,0.273958


**Conclusión:** Entre grupos de al menos 100 clientes, Fiber presenta 51.03% de abandono (4239 clientes) y No Internet presenta 27.40% (960 clientes). Los grupos pequeños requieren cautela y las diferencias no prueban causalidad. **Decisión:** mantener esta variable como candidata y comprobar su aporte en validación, considerando posibles relaciones con otras variables.

## Abandono según AutoPay
**Pregunta:** ¿la proporción de abandono cambia entre los grupos? Calculamos tamaño y tasa; no confundimos número de abandonos con riesgo relativo del grupo.

In [6]:
# dropna=False conserva a quienes no tienen información en esta variable.
summary = df.groupby('AutoPay', dropna=False).abandono.agg(['size', 'mean']).reset_index()
summary = summary.rename(columns={'size':'clientes', 'mean':'tasa'})
display(summary)
px.bar(summary, x='AutoPay', y='tasa', hover_data=['clientes'], title='Abandono según AutoPay', labels={'tasa':'Proporción de abandono'}).show()

,AutoPay,clientes,tasa
0,No,4252,0.458137
1,Yes,5748,0.386395


**Conclusión:** Entre grupos de al menos 100 clientes, No presenta 45.81% de abandono (4252 clientes) y Yes presenta 38.64% (5748 clientes). Los grupos pequeños requieren cautela y las diferencias no prueban causalidad. **Decisión:** mantener esta variable como candidata y comprobar su aporte en validación, considerando posibles relaciones con otras variables.

## Abandono según TenureCohort
**Pregunta:** ¿la proporción de abandono cambia entre los grupos? Calculamos tamaño y tasa; no confundimos número de abandonos con riesgo relativo del grupo.

In [7]:
# dropna=False conserva a quienes no tienen información en esta variable.
summary = df.groupby('TenureCohort', dropna=False).abandono.agg(['size', 'mean']).reset_index()
summary = summary.rename(columns={'size':'clientes', 'mean':'tasa'})
display(summary)
px.bar(summary, x='TenureCohort', y='tasa', hover_data=['clientes'], title='Abandono según TenureCohort', labels={'tasa':'Proporción de abandono'}).show()

,TenureCohort,clientes,tasa
0,1-3 months,95,0.757895
1,13-24 months,2248,0.543149
2,25-36 months,2128,0.454887
3,37-48 months,1553,0.360592
4,4-12 months,1084,0.600554
5,49-60 months,1028,0.286965
6,61-72 months,1864,0.215665


**Conclusión:** Entre grupos de al menos 100 clientes, 4-12 months presenta 60.06% de abandono (1084 clientes) y 61-72 months presenta 21.57% (1864 clientes). Los grupos pequeños requieren cautela y las diferencias no prueban causalidad. **Decisión:** mantener esta variable como candidata y comprobar su aporte en validación, considerando posibles relaciones con otras variables.

## Abandono según Complaints
**Pregunta:** ¿la proporción de abandono cambia entre los grupos? Calculamos tamaño y tasa; no confundimos número de abandonos con riesgo relativo del grupo.

In [8]:
# dropna=False conserva a quienes no tienen información en esta variable.
summary = df.groupby('Complaints', dropna=False).abandono.agg(['size', 'mean']).reset_index()
summary = summary.rename(columns={'size':'clientes', 'mean':'tasa'})
display(summary)
px.bar(summary, x='Complaints', y='tasa', hover_data=['clientes'], title='Abandono según Complaints', labels={'tasa':'Proporción de abandono'}).show()

,Complaints,clientes,tasa
0,0,6130,0.375693
1,1,2815,0.453996
2,2,808,0.523515
3,3,202,0.638614
4,4,34,0.852941
5,5,8,0.625000
6,6,3,0.666667


**Conclusión:** Entre grupos de al menos 100 clientes, 3.0 presenta 63.86% de abandono (202 clientes) y 0.0 presenta 37.57% (6130 clientes). Los grupos pequeños requieren cautela y las diferencias no prueban causalidad. **Decisión:** mantener esta variable como candidata y comprobar su aporte en validación, considerando posibles relaciones con otras variables.

## Abandono según CSAT
**Pregunta:** ¿la proporción de abandono cambia entre los grupos? Calculamos tamaño y tasa; no confundimos número de abandonos con riesgo relativo del grupo.

In [9]:
# dropna=False conserva a quienes no tienen información en esta variable.
summary = df.groupby('CSAT', dropna=False).abandono.agg(['size', 'mean']).reset_index()
summary = summary.rename(columns={'size':'clientes', 'mean':'tasa'})
display(summary)
px.bar(summary, x='CSAT', y='tasa', hover_data=['clientes'], title='Abandono según CSAT', labels={'tasa':'Proporción de abandono'}).show()

,CSAT,clientes,tasa
0,1.0,14,0.857143
1,2.0,639,0.500782
2,3.0,4504,0.459813
3,4.0,4288,0.369636
4,5.0,405,0.308642
5,NaN,150,0.373333


**Conclusión:** Entre grupos de al menos 100 clientes, 2.0 presenta 50.08% de abandono (639 clientes) y 5.0 presenta 30.86% (405 clientes). Los grupos pequeños requieren cautela y las diferencias no prueban causalidad. **Decisión:** mantener esta variable como candidata y comprobar su aporte en validación, considerando posibles relaciones con otras variables.

## 4. Distribuciones y valores extremos
Exploramos cargos, antigüedad y satisfacción. Los diagramas de caja señalan extremos estadísticos, no errores confirmados.

In [10]:
columns = ['MonthlyCharges','TenureMonths','TotalCharges','NPS','CSAT']
display(df[columns].describe())
for column in columns:
    # Comparamos distribuciones; no filtramos automáticamente clientes extremos.
    px.histogram(df, x=column, color='Churn', barmode='overlay', histnorm='probability density', title=f'Distribución de {column}').show()
    px.box(df, x='Churn', y=column, title=f'Rangos de {column} por abandono').show()

,MonthlyCharges,TenureMonths,TotalCharges,NPS,CSAT
count,9850.000000,10000.000000,10000.000000,9850.000000,9850.000000
mean,87.745546,36.685400,3139.692693,6.115838,3.449848
std,23.084098,20.747189,2023.934481,1.712710,0.684134
min,18.000000,1.000000,66.380000,0.000000,1.000000
25%,77.860000,19.750000,1490.980000,5.000000,3.000000
50%,91.195000,33.000000,2718.200000,6.000000,3.000000
75%,103.700000,53.000000,4513.840000,7.000000,4.000000
max,138.960000,72.000000,9744.850000,10.000000,5.000000


**Conclusiones por variable:**
- **MonthlyCharges:** rango observado 18.00–138.96; mediana 91.19. Estos límites describen la muestra, no reglas universales.
- **TenureMonths:** rango observado 1.00–72.00; mediana 33.00. Estos límites describen la muestra, no reglas universales.
- **TotalCharges:** rango observado 66.38–9744.85; mediana 2718.20. Estos límites describen la muestra, no reglas universales.
- **NPS:** rango observado 0.00–10.00; mediana 6.00. Estos límites describen la muestra, no reglas universales.
- **CSAT:** rango observado 1.00–5.00; mediana 3.00. Estos límites describen la muestra, no reglas universales.

**Decisión:** no eliminar extremos sin fundamento. Confirmar moneda y definición de NPS: una puntuación individual 0–10 no es el indicador agregado NPS.

## 5. Correlaciones y redundancia
Pearson resume relaciones lineales y Spearman relaciones de orden. Ninguna demuestra causalidad. Excluimos la etiqueta para estudiar redundancia entre predictores.

In [11]:
numeric = df.select_dtypes(include='number').drop(columns='abandono')
for method in ['pearson','spearman']:
    corr = numeric.corr(method=method)
    px.imshow(corr, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', title=f'Correlación {method}').show()

**Conclusión:** las asociaciones Spearman más fuertes son TotalCharges–EstimatedCLV: 0.981; TenureMonths–TotalCharges: 0.892; TenureMonths–EstimatedCLV: 0.826. **Decisión:** estudiar redundancia antes de añadir variables o aplicar PCA; no retirar columnas únicamente por esta tabla.

## 6. Fuga de información y disponibilidad
Comprobamos si conocer el motivo de baja equivale a conocer la etiqueta.

In [12]:
# Esta igualdad es una comprobación de fuga directa, no un modelo útil.
print('ChurnReason revela Churn:', (df.ChurnReason.ne('No churn') == df.Churn.eq('Yes')).all())

ChurnReason revela Churn: True


**Conclusión:** ChurnReason revela la etiqueta en todos los registros. Se excluye del modelo junto con CustomerID. También debemos verificar cómo se generaron EstimatedCLV, AvgTicketCost y las variables ampliadas: desconocemos si usaron información posterior al abandono. Excluir una columna no garantiza ausencia total de fuga.

## 7. Próximas decisiones
No se añade feature engineering arbitrario. Primero verificar períodos de quejas/tickets y fórmulas de valores estimados; después comparar variantes con validación cruzada. Las conclusiones descriptivas no demuestran ahorro ni eficacia de campañas.